# 📊 Thống Kê Dataset Fire/Smoke Detection (Dữ Liệu Đã Fix Pipeline)

Sổ tay này thống kê và trực quan hóa toàn bộ dữ liệu phân đoạn (Segmentation) sau khi đã xử lý các vấn đề pipeline:
1. **DS1 (Dataset_Segmentation_AutoConverted)**: Dataset ảnh tĩnh chuyển đổi từ SAM.
2. **DS2 (Dataset_Part2_Fixed)**: Dataset trích xuất từ video frames đã fix **Naming Collision** (bổ sung clip_hash), loại bỏ **False Positive clips**, lọc **Tiny Masks (< 200px²)**, và phân chia theo **Clip-Level (80/15/5)**.


In [ ]:
%matplotlib inline
import os, sys, random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import cv2
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.dpi'] = 120

# Tự động xác định project root
ROOT = Path('.').resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

print('Thư mục dự án (ROOT):', ROOT)
print('DS1 Labels:', ROOT / 'Dataset_Segmentation_AutoConverted' / 'labels')
print('DS2 Train Labels:', ROOT / 'Dataset_Part2_Fixed' / 'train' / 'labels')


In [ ]:
def collect_stats(label_dir):
    label_path = Path(label_dir)
    if not label_path.exists():
        return {'total_files': 0, 'labeled_files': 0, 'empty_files': 0, 'detection_rate': 0,
                'fire_count': 0, 'smoke_count': 0, 'total_polygons': 0, 'poly_pts': [], 'poly_areas': []}
    
    files = list(label_path.glob('*.txt'))
    non_empty = []
    c = {0: 0, 1: 0}
    poly_pts = []
    poly_areas = []
    
    for f in files:
        lines = [l for l in f.read_text(encoding='utf-8', errors='ignore').strip().splitlines() if l.strip()]
        if lines:
            non_empty.append(f)
        for line in lines:
            parts = line.strip().split()
            if len(parts) >= 7:
                cls_id = int(parts[0])
                if cls_id in c:
                    c[cls_id] += 1
                n_pts = (len(parts) - 1) // 2
                poly_pts.append(n_pts)
                coords = list(map(float, parts[1:]))
                xs = [coords[i] for i in range(0, len(coords), 2)]
                ys = [coords[i] for i in range(1, len(coords), 2)]
                if xs and ys:
                    area = (max(xs) - min(xs)) * (max(ys) - min(ys))
                    poly_areas.append(area)
                    
    total = len(files)
    labeled = len(non_empty)
    return {
        'total_files': total,
        'labeled_files': labeled,
        'empty_files': total - labeled,
        'detection_rate': labeled / max(1, total),
        'fire_count': c[0],
        'smoke_count': c[1],
        'total_polygons': c[0] + c[1],
        'poly_pts': poly_pts,
        'poly_areas': poly_areas,
    }

ds1 = collect_stats(ROOT / 'Dataset_Segmentation_AutoConverted' / 'labels')

ds2_splits = {}
for split in ['train', 'val', 'test']:
    ld = ROOT / 'Dataset_Part2_Fixed' / split / 'labels'
    if ld.exists():
        ds2_splits[split] = collect_stats(ld)

ds2_total = {
    'total_files': sum(v['total_files'] for v in ds2_splits.values()),
    'labeled_files': sum(v['labeled_files'] for v in ds2_splits.values()),
    'empty_files': sum(v['empty_files'] for v in ds2_splits.values()),
    'fire_count': sum(v['fire_count'] for v in ds2_splits.values()),
    'smoke_count': sum(v['smoke_count'] for v in ds2_splits.values()),
    'total_polygons': sum(v['total_polygons'] for v in ds2_splits.values()),
    'poly_pts': sum([v['poly_pts'] for v in ds2_splits.values()], []),
    'poly_areas': sum([v['poly_areas'] for v in ds2_splits.values()], []),
}
ds2_total['detection_rate'] = ds2_total['labeled_files'] / max(1, ds2_total['total_files'])

print('=== DS1: Dataset_Segmentation_AutoConverted ===')
print('  File nhãn      :', ds1['total_files'], f"(Có nhãn: {ds1['labeled_files']:,} - {ds1['detection_rate']*100:.1f}%)")
print('  Lửa (0)        :', ds1['fire_count'], 'polygons')
print('  Khói (1)       :', ds1['smoke_count'], 'polygons')
print('  Tổng Polygons  :', ds1['total_polygons'])

print('\n=== DS2 Fixed: Dataset_Part2_Fixed ===')
for split, stats in ds2_splits.items():
    print(f"  [{split.upper():<5}] {stats['labeled_files']:,}/{stats['total_files']:,} files | Lửa={stats['fire_count']:,}, Khói={stats['smoke_count']:,}, Tổng={stats['total_polygons']:,}")
print('  Tổng DS2 Polygons:', ds2_total['total_polygons'])


## 1. Biểu Đồ Thống Kê Tổng Quan


In [ ]:
fig = plt.figure(figsize=(18, 10))
gs = GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.32)
COLORS = {'fire': '#e74c3c', 'smoke': '#f39c12', 'ds1': '#2980b9', 'ds2': '#27ae60', 'empty': '#bdc3c7'}

ax1 = fig.add_subplot(gs[0, 0])
x = np.arange(2)
labeled_vals = [ds1['labeled_files'], ds2_total['labeled_files']]
empty_vals   = [ds1['empty_files'],   ds2_total['empty_files']]
b1 = ax1.bar(x, labeled_vals, color=[COLORS['ds1'], COLORS['ds2']], width=0.5, label='Có nhãn')
b2 = ax1.bar(x, empty_vals, bottom=labeled_vals, color=COLORS['empty'], width=0.5, label='Rỗng')
for bar, v in zip(b1, labeled_vals):
    if v > 0:
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_y() + v/2, f'{v:,}',
                 ha='center', va='center', fontweight='bold', color='white', fontsize=10)
for bar, v_lab, v_emp in zip(b2, labeled_vals, empty_vals):
    if v_emp > 0:
        ax1.text(bar.get_x() + bar.get_width()/2, v_lab + v_emp/2, f'{v_emp:,}',
                 ha='center', va='center', fontweight='bold', color='#555555', fontsize=9)
ax1.set_xticks(x)
ax1.set_xticklabels(['DS1 (AutoConv)', 'DS2 (Fixed)'], fontsize=10)
ax1.set_ylabel('Số lượng file')
ax1.set_title('Số Lượng File Ảnh & File Nhãn', fontweight='bold', fontsize=11)
ax1.legend(fontsize=9); ax1.grid(axis='y', alpha=0.3)

ax2 = fig.add_subplot(gs[0, 1])
v1, v2 = ds1['fire_count'], ds1['smoke_count']
if v1 > 0 or v2 > 0:
    wedges, texts, autotexts = ax2.pie([v1, v2], labels=['Lửa (Fire)', 'Khói (Smoke)'],
        colors=[COLORS['fire'], COLORS['smoke']], autopct='%1.1f%%', startangle=90, textprops={'fontsize': 10})
    for at in autotexts: at.set_fontweight('bold')
ax2.set_title('DS1 - Phân Phối Lớp (' + str(ds1['total_polygons']) + ' Polygons)', fontweight='bold', fontsize=11)

ax3 = fig.add_subplot(gs[0, 2])
v3, v4 = ds2_total['fire_count'], ds2_total['smoke_count']
if v3 > 0 or v4 > 0:
    wedges2, texts2, autotexts2 = ax3.pie([v3, v4], labels=['Lửa (Fire)', 'Khói (Smoke)'],
        colors=[COLORS['fire'], COLORS['smoke']], autopct='%1.1f%%', startangle=90, textprops={'fontsize': 10})
    for at in autotexts2: at.set_fontweight('bold')
ax3.set_title('DS2 Fixed - Phân Phối Lớp (' + str(ds2_total['total_polygons']) + ' Polygons)', fontweight='bold', fontsize=11)

ax4 = fig.add_subplot(gs[1, 0])
splits = list(ds2_splits.keys())
fire_v  = [ds2_splits[s]['fire_count']  for s in splits]
smoke_v = [ds2_splits[s]['smoke_count'] for s in splits]
x2 = np.arange(len(splits))
bf = ax4.bar(x2, fire_v, color=COLORS['fire'], label='Lửa', width=0.5)
bs = ax4.bar(x2, smoke_v, bottom=fire_v, color=COLORS['smoke'], label='Khói', width=0.5)
for bar, v in zip(bf, fire_v):
    if v > 0:
        ax4.text(bar.get_x() + bar.get_width()/2, bar.get_y() + v/2, f'{v:,}',
                 ha='center', va='center', color='white', fontweight='bold', fontsize=9)
for bar, fv, sv in zip(bs, fire_v, smoke_v):
    if sv > 0:
        ax4.text(bar.get_x() + bar.get_width()/2, fv + sv/2, f'{sv:,}',
                 ha='center', va='center', color='white', fontweight='bold', fontsize=9)
ax4.set_xticks(x2); ax4.set_xticklabels([s.upper() for s in splits], fontsize=10)
ax4.set_title('DS2 Fixed - Polygons theo Tập Split', fontweight='bold', fontsize=11)
ax4.legend(fontsize=9); ax4.grid(axis='y', alpha=0.3)

ax5 = fig.add_subplot(gs[1, 1])
bins = np.linspace(0, 600, 40)
if ds1['poly_pts']:
    ax5.hist(ds1['poly_pts'], bins=bins, alpha=0.6, color=COLORS['ds1'],
             label=f'DS1 (TB: {np.mean(ds1["poly_pts"]):.0f} điểm)', density=True)
if ds2_total['poly_pts']:
    ax5.hist(ds2_total['poly_pts'], bins=bins, alpha=0.6, color=COLORS['ds2'],
             label=f'DS2 (TB: {np.mean(ds2_total["poly_pts"]):.0f} điểm)', density=True)
ax5.set_xlabel('Số điểm / polygon', fontsize=9)
ax5.set_ylabel('Mật độ', fontsize=9)
ax5.set_title('Phân Phối Số Điểm Polygon', fontweight='bold', fontsize=11)
ax5.legend(fontsize=8); ax5.grid(alpha=0.3)

ax6 = fig.add_subplot(gs[1, 2])
cats = ['DS1 (AutoConv)'] + [f'DS2 {s.upper()}' for s in splits]
rates = [ds1['detection_rate']] + [ds2_splits[s]['detection_rate'] for s in splits]
bcolors = [COLORS['ds1']] + [COLORS['ds2']]*len(splits)
bars6 = ax6.bar(cats, [r*100 for r in rates], color=bcolors, width=0.5)
for bar, r in zip(bars6, rates):
    ax6.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.0,
             f'{r*100:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax6.set_ylabel('Tỷ lệ có nhãn (%)', fontsize=9)
ax6.set_ylim(0, 115)
ax6.set_title('Tỷ Lệ Frame Phát Hiện Được Vùng Khói/Lửa', fontweight='bold', fontsize=11)
ax6.grid(axis='y', alpha=0.3)

fig.suptitle('BẢNG THỐNG KÊ TỔNG QUAN DATASET FIRE/SMOKE SEGMENTATION', fontsize=14, fontweight='bold', y=1.02)
plt.show()


## 2. Bảng Tổng Hợp Chi Tiết


In [ ]:
print('=' * 88)
print(f"{'Tên Dataset / Split':<35} {'Tổng File':>10} {'Có Nhãn':>10} {'Tỷ Lệ':>8} {'Lửa (0)':>9} {'Khói (1)':>9} {'Tổng Polygons':>12}")
print('=' * 88)

print(f"{'DS1: Dataset_Segmentation_AutoConverted':<35} {ds1['total_files']:>10,} {ds1['labeled_files']:>10,} {ds1['detection_rate']*100:>7.1f}% {ds1['fire_count']:>9,} {ds1['smoke_count']:>9,} {ds1['total_polygons']:>12,}")

print('-' * 88)
for split, stats in ds2_splits.items():
    s_name = f"DS2 Fixed ({split})"
    print(f"{s_name:<35} {stats['total_files']:>10,} {stats['labeled_files']:>10,} {stats['detection_rate']*100:>7.1f}% {stats['fire_count']:>9,} {stats['smoke_count']:>9,} {stats['total_polygons']:>12,}")

print('-' * 88)
gf = ds1['fire_count']  + ds2_total['fire_count']
gs = ds1['smoke_count'] + ds2_total['smoke_count']
gt = gf + gs
tot_files = ds1['total_files'] + ds2_total['total_files']
tot_lab = ds1['labeled_files'] + ds2_total['labeled_files']
print(f"{'TỔNG CỘNG HỢP NHẤT':<35} {tot_files:>10,} {tot_lab:>10,} {tot_lab/max(1,tot_files)*100:>7.1f}% {gf:>9,} {gs:>9,} {gt:>12,}")
print('=' * 88)


## 3. Trực Quan Hóa 500 Mẫu Overlay (3 Mẫu Cho Mỗi Video Clip)


In [ ]:
import json
manifest_path = ROOT / 'Dataset_Part2_Fixed' / 'samples_3_per_video' / 'samples_manifest.json'
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    print(f"Tổng số ảnh mẫu đã trích xuất: {len(manifest):,} frames")
    
    # Nhóm các sample theo clip
    clip_samples = {}
    for item in manifest:
        clip_samples.setdefault(item['clip_name'], []).append(item)
        
    print(f"Số lượng video clips đại diện: {len(clip_samples):,} clips (mỗi clip trích xuất tối đa 3 frames)")
    
    # Hiển thị 8 clips tiêu biểu (24 frames) theo dạng lưới 3 cột (3 frames / clip)
    sample_clips = list(clip_samples.keys())[:8]
    
    fig, axes = plt.subplots(len(sample_clips), 3, figsize=(16, 3.2 * len(sample_clips)))
    if len(sample_clips) == 1:
        axes = np.array([axes])
        
    for row_idx, clip_name in enumerate(sample_clips):
        items = clip_samples[clip_name]
        for col_idx in range(3):
            ax = axes[row_idx, col_idx]
            if col_idx < len(items):
                item = items[col_idx]
                img_p = ROOT / item['clip_path']
                if img_p.exists():
                    img = cv2.cvtColor(cv2.imread(str(img_p)), cv2.COLOR_BGR2RGB)
                    ax.imshow(img)
                    ax.set_title(item['old_stem'] + '\n(' + clip_name[:25] + '...)', fontsize=8)
            ax.axis('off')
            
    fig.suptitle('Mẫu Overlay Trực Quan Hóa (3 Frames / Video Clip)', fontsize=13, fontweight='bold', y=1.002)
    plt.tight_layout()
    plt.show()
    print("500 mẫu trực quan hóa đầy đủ được lưu tại:")
    print("  - Phân loại theo clip: Dataset_Part2_Fixed/samples_3_per_video/")
    print("  - Thư mục phẳng     : Dataset_Part2_Fixed/sample_500_visualizations/")
else:
    print('Chưa tìm thấy samples_manifest.json. Vui lòng chạy generate_3_samples_per_video.py.')


## 4. Kết Luận & Đánh Giá Chất Lượng


In [ ]:
print('=' * 75)
print(' DỰ ÁN FIRE/SMOKE EARLY DETECTION — ĐÁNH GIÁ VÀ HƯỚNG DẪN TỐI ƯU')
print('=' * 75)
print(f'1. Quy mô dữ liệu sau khi sửa lỗi pipeline:')
print(f'   - Tổng số Polygons hợp nhất : {gt:,} polygons')
print(f'   - Lửa (Fire)                : {gf:,} ({gf/max(1,gt)*100:.1f}%)')
print(f'   - Khói (Smoke)              : {gs:,} ({gs/max(1,gt)*100:.1f}%)')
print()
print('2. Tình trạng phân chia dữ liệu DS2 Fixed:')
print('   - Đã loại bỏ hoàn toàn 470 frames từ các clip False-Positive (lửa lò sưởi, khói bếp)')
print('   - Đã gán md5 hash duy nhất cho từng frame theo clip ID, khắc phục hoàn toàn đè tên file')
print('   - Phân chia train/val/test theo Clip-level (501 train / 94 val / 32 test clips) -> Tránh Data Leakage')
print()
print('3. Khuyến nghị cho bài toán huấn luyện YOLOv8-seg / SegFormer:')
print('   - Do lớp Khói (Smoke) nhiều gấp ~2.5 lần lớp Lửa (Fire), nên thiết lập loss class weight:')
print('     fire_weight = 2.5 - 3.0, smoke_weight = 1.0')
print('   - Sử dụng Dataset_Part2_Fixed/data.yaml để fine-tune mô hình YOLOv8s-seg.')
print('=' * 75)


## 5. Kết Quả Suy Luận Phân Đoạn (YOLOv8x-seg Auto-Annotator Inference)


In [ ]:
import json
manifest_path = ROOT / 'reports' / 'yolov8x_seg_inference_manifest.json'
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    print('Tổng số ảnh đã suy luận (Inference):', len(manifest))
    
    tot_det = sum(item['num_detections'] for item in manifest)
    c0 = sum(item['classes'].count(0) for item in manifest)
    c1 = sum(item['classes'].count(1) for item in manifest)
    print('Tổng số polygons phát hiện:', tot_det)
    print('  - Lửa (Fire) :', c0)
    print('  - Khói (Smoke):', c1)
    
    det_items = [item for item in manifest if item['num_detections'] > 0]
    if not det_items:
        det_items = manifest
        
    random.seed(42)
    sample_items = random.sample(det_items, min(6, len(det_items)))
    
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    for ax, item in zip(axes.flatten(), sample_items):
        img_p = ROOT / item['out_path']
        if img_p.exists():
            img_rgb = cv2.cvtColor(cv2.imread(str(img_p)), cv2.COLOR_BGR2RGB)
            ax.imshow(img_rgb)
            ax.set_title(item['orig_name'][:30] + '... (' + str(item['num_detections']) + ' objs)', fontsize=8)
        ax.axis('off')
        
    for ax in axes.flatten()[len(sample_items):]:
        ax.axis('off')
        
    fig.suptitle('Kết Quả Phân Đoạn Trực Quan Với Mô Hình YOLOv8x-seg Auto-Annotator', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print('Thư mục lưu 500 ảnh suy luận đầy đủ: reports/yolov8x_seg_inference_500/')
else:
    print('Chưa tìm thấy yolov8x_seg_inference_manifest.json.')
